In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"D:\Big Data Programming Project\Final Assignment"
)

SIRIVM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "sirivm"
)

TIMETABLE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "timetable"
)

SERVICE_DATES = [
    "2025-12-26",
    "2025-12-27",
    "2025-12-28"
]

for service_date in SERVICE_DATES:
    siri = pd.read_csv(
        SIRIVM_ROOT
        / service_date
        / "scne_fresh_sirivm.csv",
        dtype=str
    )

    trips = pd.read_csv(
        TIMETABLE_ROOT
        / service_date
        / "scne_active_trips.csv",
        dtype=str
    )

    stop_times = pd.read_csv(
        TIMETABLE_ROOT
        / service_date
        / "scne_active_stop_times.csv",
        dtype=str
    )

    stops = pd.read_csv(
        TIMETABLE_ROOT
        / service_date
        / "scne_active_stops.csv",
        dtype=str
    )

    print(f"\nDate: {service_date}")
    print("SIRI-VM rows:", len(siri))
    print("Trips:", len(trips))
    print("Stop-time rows:", len(stop_times))
    print("Stops:", len(stops))


Date: 2025-12-26
SIRI-VM rows: 85589
Trips: 933
Stop-time rows: 42444
Stops: 1771

Date: 2025-12-27
SIRI-VM rows: 495255
Trips: 5004
Stop-time rows: 220592
Stops: 3992

Date: 2025-12-28
SIRI-VM rows: 279671
Trips: 3088
Stop-time rows: 135017
Stops: 3501


In [2]:
service_date = "2025-12-27"

siri = pd.read_csv(
    SIRIVM_ROOT / service_date / "scne_fresh_sirivm.csv",
    dtype=str
)

trips = pd.read_csv(
    TIMETABLE_ROOT / service_date / "scne_active_trips.csv",
    dtype=str
)

print("SIRI journey fields:")
print(
    siri[
        [
            "line_ref",
            "published_line_name",
            "direction_ref",
            "dated_journey_ref",
            "vehicle_journey_ref",
            "origin_aimed_departure_time"
        ]
    ].head(10)
)

print("\nGTFS journey fields:")
print(
    trips[
        [
            "route_id",
            "trip_id",
            "direction_id",
            "vehicle_journey_code"
        ]
    ].head(10)
)

SIRI journey fields:
  line_ref published_line_name direction_ref dated_journey_ref  \
0      787                 787      outbound                 1   
1      787                 787      outbound                 1   
2      787                 787      outbound                 1   
3      787                 787      outbound                 1   
4      787                 787      outbound                 1   
5      787                 787      outbound                 1   
6      787                 787      outbound                 1   
7      787                 787      outbound                 1   
8      787                 787      outbound                 1   
9      787                 787      outbound                 1   

  vehicle_journey_ref origin_aimed_departure_time  
0                 NaN   2025-12-27T02:50:00+00:00  
1                 NaN   2025-12-27T02:50:00+00:00  
2                 NaN   2025-12-27T02:50:00+00:00  
3                 NaN   2025-12-27T02:50:00+

In [3]:
import zipfile

zip_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "timetable"
    / service_date
    / "itm_north_east_gtfs_20251227.zip"
)

with zipfile.ZipFile(zip_path, "r") as archive:
    routes = pd.read_csv(
        archive.open("routes.txt"),
        dtype=str
    )

trips = pd.read_csv(
    TIMETABLE_ROOT / service_date / "scne_active_trips.csv",
    dtype=str
)

stop_times = pd.read_csv(
    TIMETABLE_ROOT / service_date / "scne_active_stop_times.csv",
    dtype=str
)

# Add route number to trips
trips = trips.merge(
    routes[["route_id", "route_short_name"]],
    on="route_id",
    how="left"
)

# Get first scheduled stop for each trip
stop_times["stop_sequence_num"] = pd.to_numeric(
    stop_times["stop_sequence"],
    errors="coerce"
)

first_stops = (
    stop_times
    .sort_values(["trip_id", "stop_sequence_num"])
    .groupby("trip_id")
    .first()
    .reset_index()
)

trips = trips.merge(
    first_stops[
        ["trip_id", "departure_time"]
    ],
    on="trip_id",
    how="left"
)

# SIRI scheduled origin time -> HH:MM
siri["origin_time"] = pd.to_datetime(
    siri["origin_aimed_departure_time"],
    utc=True,
    errors="coerce"
).dt.strftime("%H:%M")

# GTFS scheduled origin time -> HH:MM
trips["origin_time"] = (
    trips["departure_time"]
    .str[:5]
)

print("SIRI unique journeys:",
      siri[
          ["published_line_name",
           "direction_ref",
           "origin_time"]
      ].drop_duplicates().shape[0])

print("GTFS active trips:", len(trips))

print("\nSIRI directions:")
print(siri["direction_ref"].value_counts())

print("\nGTFS directions:")
print(trips["direction_id"].value_counts())

SIRI unique journeys: 4825
GTFS active trips: 5004

SIRI directions:
direction_ref
outbound    277265
inbound     217990
Name: count, dtype: int64

GTFS directions:
direction_id
0    2787
1    2217
Name: count, dtype: int64


In [4]:
# Build unique SIRI journey keys
siri_journeys = siri[
    [
        "published_line_name",
        "direction_ref",
        "origin_time"
    ]
].drop_duplicates().copy()

# Test both possible direction mappings
mapping_1 = {
    "outbound": "0",
    "inbound": "1"
}

mapping_2 = {
    "outbound": "1",
    "inbound": "0"
}

for name, mapping in [
    ("Mapping 1", mapping_1),
    ("Mapping 2", mapping_2)
]:
    test_siri = siri_journeys.copy()

    test_siri["direction_id"] = (
        test_siri["direction_ref"].map(mapping)
    )

    matches = test_siri.merge(
        trips[
            [
                "route_short_name",
                "direction_id",
                "origin_time",
                "trip_id"
            ]
        ],
        left_on=[
            "published_line_name",
            "direction_id",
            "origin_time"
        ],
        right_on=[
            "route_short_name",
            "direction_id",
            "origin_time"
        ],
        how="inner"
    )

    matched_journeys = matches[
        [
            "published_line_name",
            "direction_ref",
            "origin_time"
        ]
    ].drop_duplicates().shape[0]

    print(
        name,
        "| Matched SIRI journeys:",
        matched_journeys,
        "of",
        len(siri_journeys)
    )

Mapping 1 | Matched SIRI journeys: 4825 of 4825
Mapping 2 | Matched SIRI journeys: 583 of 4825


In [5]:
direction_map = {
    "outbound": "0",
    "inbound": "1"
}

siri["direction_id"] = (
    siri["direction_ref"]
    .map(direction_map)
)

matched_siri = siri.merge(
    trips[
        [
            "trip_id",
            "route_id",
            "route_short_name",
            "direction_id",
            "origin_time"
        ]
    ],
    left_on=[
        "published_line_name",
        "direction_id",
        "origin_time"
    ],
    right_on=[
        "route_short_name",
        "direction_id",
        "origin_time"
    ],
    how="left"
)

print("SIRI rows:", len(siri))
print("Matched rows:", matched_siri["trip_id"].notna().sum())
print("Unmatched rows:", matched_siri["trip_id"].isna().sum())
print("Unique matched trips:", matched_siri["trip_id"].nunique())

SIRI rows: 495255
Matched rows: 515668
Unmatched rows: 0
Unique matched trips: 4921


In [6]:
gtfs_key_counts = (
    trips.groupby(
        [
            "route_short_name",
            "direction_id",
            "origin_time"
        ]
    )
    .size()
    .reset_index(name="trip_count")
)

ambiguous_keys = gtfs_key_counts[
    gtfs_key_counts["trip_count"] > 1
].copy()

print("Total GTFS journey keys:", len(gtfs_key_counts))
print("Ambiguous journey keys:", len(ambiguous_keys))
print(
    "GTFS trips involved in ambiguous keys:",
    ambiguous_keys["trip_count"].sum()
)

display(
    ambiguous_keys
    .sort_values("trip_count", ascending=False)
    .head(20)
)

Total GTFS journey keys: 4908
Ambiguous journey keys: 91
GTFS trips involved in ambiguous keys: 187


,route_short_name,direction_id,origin_time,trip_count
2651,38,1,12:15,3
2654,38,1,13:15,3
2657,38,1,14:15,3
2660,38,1,15:15,3
2648,38,1,11:15,3
92,1,0,15:37,2
90,1,0,15:22,2
17,1,0,09:17,2
11,1,0,08:02,2
13,1,0,08:32,2


In [7]:
ambiguous_trip_keys = ambiguous_keys[
    ["route_short_name", "direction_id", "origin_time"]
]

ambiguous_trips = trips.merge(
    ambiguous_trip_keys,
    on=[
        "route_short_name",
        "direction_id",
        "origin_time"
    ],
    how="inner"
)

print(
    "Ambiguous GTFS trips:",
    len(ambiguous_trips)
)

print(
    "Unique trip headsigns:",
    ambiguous_trips["trip_headsign"].nunique()
)

display(
    ambiguous_trips[
        [
            "route_short_name",
            "direction_id",
            "origin_time",
            "trip_id",
            "trip_headsign",
            "vehicle_journey_code"
        ]
    ].head(30)
)

Ambiguous GTFS trips: 187
Unique trip headsigns: 39


,route_short_name,direction_id,origin_time,trip_id,trip_headsign,vehicle_journey_code
0,1,1,21:56,VJ0ea7b01989c078897f9d6b4726a25f4cd08b4d95,Wainstones Walk,VJ1209
1,1,1,19:56,VJ7c3acd73e965b1138c37df3e95eecbe9e68b6011,Ryelands Park,VJ1205
2,12,0,11:00,VJ1b879c53a0502e68b5ee8cdac927c50ac715780a,Sth Shields Interchange,VJ280
3,12,0,09:30,VJ89533e95c4e8db7d55cf16c33c9a4635f63c564f,Sth Shields Interchange,VJ276
4,12,0,10:00,VJa547881bb4d91441887a4b1cb3d27af455d5b40b,Sth Shields Interchange,VJ278
5,12,0,12:00,VJd4b5d9bf00184fc9467b8566d748edf31c2e1641,Sth Shields Interchange,VJ282
6,12,0,09:30,VJ4e18b3acc346bceaa27f57619a12b8b8ee8a2139,Coulby Newham,VJ1493
7,12,0,15:25,VJ810c7ffa957fad0393b7b90e215903cb353c0265,Coulby Newham,VJ1535
8,12,0,07:40,VJ9843e0353159b1af01d8576681cde3718abe5200,Coulby Newham,VJ1484
9,12,0,12:00,VJb62fc799b45425e676045fbb725e6142f836a9d0,Coulby Newham,VJ1513


In [8]:
# Prepare unique SIRI journey records
siri_journeys = siri[
    [
        "published_line_name",
        "direction_ref",
        "origin_time",
        "destination_name"
    ]
].drop_duplicates().copy()

siri_journeys["direction_id"] = (
    siri_journeys["direction_ref"]
    .map({
        "outbound": "0",
        "inbound": "1"
    })
)

# Merge using route + direction + origin time
candidate_matches = siri_journeys.merge(
    trips[
        [
            "trip_id",
            "route_short_name",
            "direction_id",
            "origin_time",
            "trip_headsign"
        ]
    ],
    left_on=[
        "published_line_name",
        "direction_id",
        "origin_time"
    ],
    right_on=[
        "route_short_name",
        "direction_id",
        "origin_time"
    ],
    how="left"
)

# Check destination/headsign equality
candidate_matches["destination_match"] = (
    candidate_matches["destination_name"]
    .str.lower()
    .str.strip()
    ==
    candidate_matches["trip_headsign"]
    .str.lower()
    .str.strip()
)

print("Candidate rows:", len(candidate_matches))
print("Exact destination/headsign matches:",
      candidate_matches["destination_match"].sum())

print("\nAmbiguous candidates with destination match:")
display(
    candidate_matches[
        candidate_matches["destination_match"]
    ][
        [
            "published_line_name",
            "direction_ref",
            "origin_time",
            "destination_name",
            "trip_headsign",
            "trip_id"
        ]
    ].head(20)
)

Candidate rows: 5055
Exact destination/headsign matches: 138

Ambiguous candidates with destination match:


,published_line_name,direction_ref,origin_time,destination_name,trip_headsign,trip_id
82,1,outbound,06:55,Station,Station,VJ404cf783c9eeded252ae216e7399b775ccbd1219
89,2,outbound,06:14,Station,Station,VJ1323e3ff8af40db096ad7472a7560b113d394f1b
169,1,inbound,06:47,Ryelands Park,Ryelands Park,VJ32a30ded215800c2868c36be0a6148d45e8c7f0d
191,63,outbound,06:54,Thistle Way,Thistle Way,VJ9c058432514c863ca24e58953dcbf7534a200bda
252,59,inbound,07:12,Bus Station Stand K,Bus Station stand K,VJde53287bcffbae6d41cd6efa847534f555ee1aa2
261,10,outbound,07:17,Jarrow Bus Station,Jarrow Bus Station,VJb0eb99dd06f6a9e5cf41eff415adc202c8f8aad1
289,63,outbound,07:20,Thistle Way,Thistle Way,VJ43969c1221826d178ecfef5c87a2a4982dd00af4
349,X40,outbound,07:50,Amazon,Amazon,VJ6a6678385017aab3a5c5462b35f3a928532b1763
367,2,outbound,07:30,Station,Station,VJb02d22b4d30333d6397fdffe36dc018de163a94b
411,1,inbound,08:01,Ryelands Park,Ryelands Park,VJ9868a080b28a937cf4a83d1f9036e83d2492521c


In [9]:
# Prepare GTFS journey code without "VJ"
trips["journey_code_clean"] = (
    trips["vehicle_journey_code"]
    .str.replace("VJ", "", regex=False)
    .str.strip()
)

# Prepare SIRI journey reference
siri["journey_ref_clean"] = (
    siri["dated_journey_ref"]
    .astype(str)
    .str.strip()
)

# Test direct journey-reference matches
journey_ref_matches = siri[
    [
        "published_line_name",
        "direction_ref",
        "origin_time",
        "journey_ref_clean"
    ]
].drop_duplicates().merge(
    trips[
        [
            "route_short_name",
            "direction_id",
            "origin_time",
            "journey_code_clean",
            "trip_id"
        ]
    ],
    left_on="journey_ref_clean",
    right_on="journey_code_clean",
    how="inner"
)

print(
    "SIRI unique journeys:",
    siri[
        [
            "published_line_name",
            "direction_ref",
            "origin_time"
        ]
    ].drop_duplicates().shape[0]
)

print(
    "Journeys matching by journey reference:",
    journey_ref_matches[
        "trip_id"
    ].nunique()
)

display(journey_ref_matches.head(20))

SIRI unique journeys: 4825
Journeys matching by journey reference: 1418


,published_line_name,direction_ref,origin_time_x,journey_ref_clean,route_short_name,direction_id,origin_time_y,journey_code_clean,trip_id
0,2,outbound,04:53,551,36,0,12:40,551,VJb86408cbeed05b4a0e92b291baa577a7151d9d3f
1,36A,inbound,04:55,606,3,0,07:51,606,VJ26201035f1fb4373e76d575357ae17f9ef0fc93c
2,36A,inbound,04:55,606,38,1,17:20,606,VJaf1465de470e1732382fce2d0b48df55f3881bbd
3,36A,inbound,04:55,606,7,1,06:59,606,VJ67c35b76ad06e393c45c91597f9f27c33420dcf9
4,3,inbound,05:00,202,X24,0,19:24,202,VJ5a9ef35c2c0d967922158e596e2465b49a6694fb
5,3,inbound,05:00,202,11,1,16:05,202,VJ7e04e9b867dd846a592095b624fa5f1b55ee3faa
6,36A,outbound,05:08,611,4,0,08:39,611,VJ1345b6e730395860c1342c025dbf365a2fa8ec2d
7,36A,outbound,05:08,611,36,0,17:45,611,VJ95aa5bec98c145031fda2308d5779d626465e821
8,36A,outbound,05:08,611,7,0,08:03,611,VJ21f560169cce6e123e488fc2f8b04f4d1c7552da
9,4,outbound,05:06,201,X24,1,18:44,201,VJ4a5f25b6233ddabe526a94e046ccad4f906220eb


In [10]:
# Get first and last scheduled stop for every GTFS trip
ordered_stops = stop_times.copy()

ordered_stops["stop_sequence_num"] = pd.to_numeric(
    ordered_stops["stop_sequence"],
    errors="coerce"
)

ordered_stops = ordered_stops.sort_values(
    ["trip_id", "stop_sequence_num"]
)

trip_endpoints = (
    ordered_stops
    .groupby("trip_id")
    .agg(
        gtfs_origin_stop=("stop_id", "first"),
        gtfs_destination_stop=("stop_id", "last")
    )
    .reset_index()
)

# Add endpoints to GTFS trips
trips_with_stops = trips.merge(
    trip_endpoints,
    on="trip_id",
    how="left"
)

# Unique SIRI journeys
siri_journeys = siri[
    [
        "published_line_name",
        "direction_ref",
        "origin_time",
        "origin_ref",
        "destination_ref"
    ]
].drop_duplicates().copy()

siri_journeys["direction_id"] = (
    siri_journeys["direction_ref"]
    .map({
        "outbound": "0",
        "inbound": "1"
    })
)

# Candidate matches
candidates = siri_journeys.merge(
    trips_with_stops[
        [
            "trip_id",
            "route_short_name",
            "direction_id",
            "origin_time",
            "gtfs_origin_stop",
            "gtfs_destination_stop"
        ]
    ],
    left_on=[
        "published_line_name",
        "direction_id",
        "origin_time"
    ],
    right_on=[
        "route_short_name",
        "direction_id",
        "origin_time"
    ],
    how="left"
)

candidates["origin_match"] = (
    candidates["origin_ref"]
    == candidates["gtfs_origin_stop"]
)

candidates["destination_match"] = (
    candidates["destination_ref"]
    == candidates["gtfs_destination_stop"]
)

candidates["both_stops_match"] = (
    candidates["origin_match"]
    & candidates["destination_match"]
)

print("Candidate rows:", len(candidates))
print("Origin stop matches:", candidates["origin_match"].sum())
print("Destination stop matches:", candidates["destination_match"].sum())
print("Both origin + destination match:",
      candidates["both_stops_match"].sum())


Candidate rows: 5105
Origin stop matches: 4914
Destination stop matches: 4967
Both origin + destination match: 4914


In [11]:
matched_candidates = candidates[
    candidates["both_stops_match"]
].copy()

journey_key = [
    "published_line_name",
    "direction_ref",
    "origin_time",
    "origin_ref",
    "destination_ref"
]

match_counts = (
    matched_candidates
    .groupby(journey_key)
    ["trip_id"]
    .nunique()
    .reset_index(name="trip_matches")
)

print("SIRI journeys:", len(siri_journeys))

print(
    "Journeys with exactly 1 GTFS match:",
    (match_counts["trip_matches"] == 1).sum()
)

print(
    "Journeys with multiple GTFS matches:",
    (match_counts["trip_matches"] > 1).sum()
)

print(
    "Journeys with no endpoint match:",
    len(siri_journeys) - len(match_counts)
)

SIRI journeys: 4914
Journeys with exactly 1 GTFS match: 4914
Journeys with multiple GTFS matches: 0
Journeys with no endpoint match: 0


In [14]:
# Add route_id into the candidate table
candidates = siri_journeys.merge(
    trips_with_stops[
        [
            "trip_id",
            "route_id",
            "route_short_name",
            "direction_id",
            "origin_time",
            "gtfs_origin_stop",
            "gtfs_destination_stop"
        ]
    ],
    left_on=[
        "published_line_name",
        "direction_id",
        "origin_time"
    ],
    right_on=[
        "route_short_name",
        "direction_id",
        "origin_time"
    ],
    how="left"
)

candidates["both_stops_match"] = (
    (candidates["origin_ref"] == candidates["gtfs_origin_stop"])
    &
    (candidates["destination_ref"] == candidates["gtfs_destination_stop"])
)

matched_candidates = candidates[
    candidates["both_stops_match"]
].copy()

journey_map = matched_candidates[
    journey_key + ["trip_id", "route_id"]
].drop_duplicates()

print("Confirmed journey mappings:", len(journey_map))

Confirmed journey mappings: 4914


In [15]:
siri["direction_id"] = siri["direction_ref"].map({
    "outbound": "0",
    "inbound": "1"
})

matched_siri = siri.merge(
    journey_map,
    on=[
        "published_line_name",
        "direction_ref",
        "origin_time",
        "origin_ref",
        "destination_ref"
    ],
    how="left"
)

print("Original SIRI rows:", len(siri))
print("Matched rows:", matched_siri["trip_id"].notna().sum())
print("Unmatched rows:", matched_siri["trip_id"].isna().sum())
print("Row count unchanged:", len(matched_siri) == len(siri))
print("Unique matched trips:", matched_siri["trip_id"].nunique())

Original SIRI rows: 495255
Matched rows: 495255
Unmatched rows: 0
Row count unchanged: True
Unique matched trips: 4914


In [16]:
# Add stop coordinates to scheduled stop times
scheduled_stops = stop_times.merge(
    stops[
        [
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ]
    ],
    on="stop_id",
    how="left"
)

# Convert coordinates and stop sequence to numbers
scheduled_stops["stop_lat"] = pd.to_numeric(
    scheduled_stops["stop_lat"],
    errors="coerce"
)

scheduled_stops["stop_lon"] = pd.to_numeric(
    scheduled_stops["stop_lon"],
    errors="coerce"
)

scheduled_stops["stop_sequence"] = pd.to_numeric(
    scheduled_stops["stop_sequence"],
    errors="coerce"
)

print("Scheduled stop rows:", len(scheduled_stops))
print(
    "Missing stop coordinates:",
    scheduled_stops["stop_lat"].isna().sum()
    + scheduled_stops["stop_lon"].isna().sum()
)
print(
    "Unique trips:",
    scheduled_stops["trip_id"].nunique()
)

Scheduled stop rows: 220592
Missing stop coordinates: 15420
Unique trips: 5004


In [17]:
service_date = "2025-12-27"

stops = pd.read_csv(
    TIMETABLE_ROOT
    / service_date
    / "scne_active_stops.csv",
    dtype=str
)

scheduled_stops = stop_times.merge(
    stops[
        [
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ]
    ],
    on="stop_id",
    how="left"
)

scheduled_stops["stop_lat"] = pd.to_numeric(
    scheduled_stops["stop_lat"],
    errors="coerce"
)

scheduled_stops["stop_lon"] = pd.to_numeric(
    scheduled_stops["stop_lon"],
    errors="coerce"
)

scheduled_stops["stop_sequence"] = pd.to_numeric(
    scheduled_stops["stop_sequence"],
    errors="coerce"
)

print("Scheduled stop rows:", len(scheduled_stops))
print(
    "Missing stop coordinates:",
    scheduled_stops["stop_lat"].isna().sum()
    + scheduled_stops["stop_lon"].isna().sum()
)
print("Unique trips:", scheduled_stops["trip_id"].nunique())

Scheduled stop rows: 220592
Missing stop coordinates: 0
Unique trips: 5004


In [18]:
import numpy as np
from scipy.spatial import cKDTree

# Convert SIRI coordinates
matched_siri["latitude"] = pd.to_numeric(
    matched_siri["latitude"],
    errors="coerce"
)

matched_siri["longitude"] = pd.to_numeric(
    matched_siri["longitude"],
    errors="coerce"
)

matched_parts = []

for trip_id, siri_trip in matched_siri.groupby("trip_id"):
    gtfs_trip = scheduled_stops[
        scheduled_stops["trip_id"] == trip_id
    ]

    if gtfs_trip.empty:
        continue

    # Build nearest-stop search tree
    tree = cKDTree(
        gtfs_trip[["stop_lat", "stop_lon"]].values
    )

    distances, indexes = tree.query(
        siri_trip[["latitude", "longitude"]].values,
        k=1
    )

    siri_trip = siri_trip.copy()

    nearest_stops = gtfs_trip.iloc[indexes]

    siri_trip["nearest_stop_id"] = (
        nearest_stops["stop_id"].values
    )

    siri_trip["nearest_stop_sequence"] = (
        nearest_stops["stop_sequence"].values
    )

    # Approximate distance in metres
    lat_diff = (
        siri_trip["latitude"].values
        - nearest_stops["stop_lat"].values
    )

    lon_diff = (
        siri_trip["longitude"].values
        - nearest_stops["stop_lon"].values
    )

    mean_lat = np.radians(
        (
            siri_trip["latitude"].values
            + nearest_stops["stop_lat"].values
        ) / 2
    )

    distance_m = np.sqrt(
        (lat_diff * 111320) ** 2
        +
        (
            lon_diff
            * 111320
            * np.cos(mean_lat)
        ) ** 2
    )

    siri_trip["stop_distance_m"] = distance_m

    matched_parts.append(siri_trip)

siri_stop_matches = pd.concat(
    matched_parts,
    ignore_index=True
)

print("SIRI observations matched:", len(siri_stop_matches))

print("\nNearest-stop distance percentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(
        f"{p}%:",
        round(
            np.percentile(
                siri_stop_matches["stop_distance_m"],
                p
            ),
            1
        ),
        "metres"
    )

print(
    "\nWithin 100 m:",
    (siri_stop_matches["stop_distance_m"] <= 100).sum()
)

print(
    "Within 150 m:",
    (siri_stop_matches["stop_distance_m"] <= 150).sum()
)

print(
    "Within 200 m:",
    (siri_stop_matches["stop_distance_m"] <= 200).sum()
)

SIRI observations matched: 495255

Nearest-stop distance percentiles:
25%: 18.5 metres
50%: 56.2 metres
75%: 123.6 metres
90%: 240.2 metres
95%: 626.8 metres
99%: 6121367.3 metres

Within 100 m: 337094
Within 150 m: 401072
Within 200 m: 433588


In [19]:
valid_coordinate_mask = (
    matched_siri["latitude"].between(49, 61)
    &
    matched_siri["longitude"].between(-8, 2)
)

print("Total SIRI rows:", len(matched_siri))
print("Valid UK coordinates:", valid_coordinate_mask.sum())
print("Invalid coordinates:", (~valid_coordinate_mask).sum())

print("\nExamples of invalid coordinates:")
display(
    matched_siri.loc[
        ~valid_coordinate_mask,
        ["latitude", "longitude", "vehicle_ref"]
    ].head(20)
)

Total SIRI rows: 495255
Valid UK coordinates: 481228
Invalid coordinates: 14027

Examples of invalid coordinates:


,latitude,longitude,vehicle_ref
2187,0.0,0.0,SCNE-36476
2217,0.0,0.0,SCNE-36476
2247,0.0,0.0,SCNE-36476
2278,0.0,0.0,SCNE-36476
2309,0.0,0.0,SCNE-36476
2340,0.0,0.0,SCNE-36476
2371,0.0,0.0,SCNE-36476
2403,0.0,0.0,SCNE-36476
2435,0.0,0.0,SCNE-36476
2468,0.0,0.0,SCNE-36476


In [20]:
clean_siri = matched_siri[
    matched_siri["latitude"].between(49, 61)
    &
    matched_siri["longitude"].between(-8, 2)
].copy()

print("Rows before coordinate cleaning:", len(matched_siri))
print("Rows after coordinate cleaning:", len(clean_siri))
print("Rows removed:", len(matched_siri) - len(clean_siri))

Rows before coordinate cleaning: 495255
Rows after coordinate cleaning: 481228
Rows removed: 14027


In [21]:
matched_parts = []

for trip_id, siri_trip in clean_siri.groupby("trip_id"):
    gtfs_trip = scheduled_stops[
        scheduled_stops["trip_id"] == trip_id
    ]

    if gtfs_trip.empty:
        continue

    tree = cKDTree(
        gtfs_trip[["stop_lat", "stop_lon"]].values
    )

    distances, indexes = tree.query(
        siri_trip[["latitude", "longitude"]].values,
        k=1
    )

    nearest_stops = gtfs_trip.iloc[indexes]

    siri_trip = siri_trip.copy()

    siri_trip["nearest_stop_id"] = (
        nearest_stops["stop_id"].values
    )

    siri_trip["nearest_stop_sequence"] = (
        nearest_stops["stop_sequence"].values
    )

    lat_diff = (
        siri_trip["latitude"].values
        - nearest_stops["stop_lat"].values
    )

    lon_diff = (
        siri_trip["longitude"].values
        - nearest_stops["stop_lon"].values
    )

    mean_lat = np.radians(
        (
            siri_trip["latitude"].values
            + nearest_stops["stop_lat"].values
        ) / 2
    )

    siri_trip["stop_distance_m"] = np.sqrt(
        (lat_diff * 111320) ** 2
        +
        (
            lon_diff
            * 111320
            * np.cos(mean_lat)
        ) ** 2
    )

    matched_parts.append(siri_trip)

siri_stop_matches = pd.concat(
    matched_parts,
    ignore_index=True
)

for p in [50, 75, 90, 95, 99]:
    print(
        f"{p}%:",
        round(
            np.percentile(
                siri_stop_matches["stop_distance_m"],
                p
            ),
            1
        ),
        "metres"
    )

print("Within 100 m:",
      (siri_stop_matches["stop_distance_m"] <= 100).sum())

print("Within 150 m:",
      (siri_stop_matches["stop_distance_m"] <= 150).sum())

print("Within 200 m:",
      (siri_stop_matches["stop_distance_m"] <= 200).sum())

50%: 53.2 metres
75%: 115.7 metres
90%: 198.7 metres
95%: 308.4 metres
99%: 1103.7 metres
Within 100 m: 337094
Within 150 m: 401072
Within 200 m: 433588


In [22]:
MAX_STOP_DISTANCE_M = 200

reliable_stop_matches = siri_stop_matches[
    siri_stop_matches["stop_distance_m"] <= MAX_STOP_DISTANCE_M
].copy()

print("Rows before stop-distance filter:", len(siri_stop_matches))
print("Rows after stop-distance filter:", len(reliable_stop_matches))
print("Rows removed:", len(siri_stop_matches) - len(reliable_stop_matches))

print(
    "Retention rate:",
    round(
        len(reliable_stop_matches)
        / len(siri_stop_matches)
        * 100,
        2
    ),
    "%"
)

Rows before stop-distance filter: 481228
Rows after stop-distance filter: 433588
Rows removed: 47640
Retention rate: 90.1 %


In [23]:
# Sort so the closest GPS observation to each stop comes first
reliable_stop_matches = reliable_stop_matches.sort_values(
    [
        "trip_id",
        "nearest_stop_sequence",
        "stop_distance_m"
    ]
)

# Keep one best observation for each journey-stop
journey_stop_events = (
    reliable_stop_matches
    .drop_duplicates(
        subset=[
            "trip_id",
            "nearest_stop_sequence"
        ],
        keep="first"
    )
    .copy()
)

# Convert observation time
journey_stop_events["actual_observed_time"] = pd.to_datetime(
    journey_stop_events["recorded_at_time"],
    utc=True,
    errors="coerce"
)

print("Reliable GPS observations:", len(reliable_stop_matches))
print("Unique journey-stop events:", len(journey_stop_events))
print(
    "Unique trips represented:",
    journey_stop_events["trip_id"].nunique()
)
print(
    "Missing observed times:",
    journey_stop_events["actual_observed_time"].isna().sum()
)
print(
    "Duplicate journey-stop events:",
    journey_stop_events.duplicated(
        subset=["trip_id", "nearest_stop_sequence"]
    ).sum()
)

Reliable GPS observations: 433588
Unique journey-stop events: 173689
Unique trips represented: 4726
Missing observed times: 0
Duplicate journey-stop events: 0


In [24]:
from datetime import timedelta

# Keep the scheduled arrival time for each trip-stop
scheduled_lookup = scheduled_stops[
    [
        "trip_id",
        "stop_sequence",
        "stop_id",
        "arrival_time"
    ]
].copy()

journey_stop_events = journey_stop_events.merge(
    scheduled_lookup,
    left_on=["trip_id", "nearest_stop_sequence"],
    right_on=["trip_id", "stop_sequence"],
    how="left"
)

# Convert GTFS HH:MM:SS to a real datetime
def make_scheduled_datetime(service_date, gtfs_time):
    if pd.isna(gtfs_time):
        return pd.NaT

    hours, minutes, seconds = map(
        int,
        gtfs_time.split(":")
    )

    base_date = pd.Timestamp(
        service_date,
        tz="UTC"
    )

    return (
        base_date
        + timedelta(
            hours=hours,
            minutes=minutes,
            seconds=seconds
        )
    )

journey_stop_events["scheduled_arrival_time"] = (
    journey_stop_events.apply(
        lambda row: make_scheduled_datetime(
            "2025-12-27",
            row["arrival_time"]
        ),
        axis=1
    )
)

journey_stop_events["delay_seconds"] = (
    journey_stop_events["actual_observed_time"]
    - journey_stop_events["scheduled_arrival_time"]
).dt.total_seconds()

print("Journey-stop rows:", len(journey_stop_events))
print(
    "Missing scheduled arrivals:",
    journey_stop_events["scheduled_arrival_time"].isna().sum()
)
print(
    "Missing delay values:",
    journey_stop_events["delay_seconds"].isna().sum()
)

print("\nDelay summary:")
print(
    journey_stop_events["delay_seconds"]
    .describe()
    .round(2)
)

Journey-stop rows: 173689
Missing scheduled arrivals: 0
Missing delay values: 0

Delay summary:
count    173689.00
mean         57.75
std         419.66
min      -12638.00
25%         -38.00
50%          25.00
75%         114.00
max       10591.00
Name: delay_seconds, dtype: float64


In [25]:
MIN_DELAY_SECONDS = -900
MAX_DELAY_SECONDS = 1800

clean_delay_data = journey_stop_events[
    journey_stop_events["delay_seconds"].between(
        MIN_DELAY_SECONDS,
        MAX_DELAY_SECONDS
    )
].copy()

print("Rows before delay filter:", len(journey_stop_events))
print("Rows after delay filter:", len(clean_delay_data))
print("Rows removed:", len(journey_stop_events) - len(clean_delay_data))

print("\nClean delay summary:")
print(
    clean_delay_data["delay_seconds"]
    .describe()
    .round(2)
)

Rows before delay filter: 173689
Rows after delay filter: 170149
Rows removed: 3540

Clean delay summary:
count    170149.00
mean         60.21
std         199.03
min        -899.00
25%         -35.00
50%          26.00
75%         112.00
max        1800.00
Name: delay_seconds, dtype: float64


In [26]:
from scipy.spatial import cKDTree
import numpy as np
import zipfile
from datetime import timedelta

CLEAN_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "delay_data"
)

CLEAN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

direction_map = {
    "outbound": "0",
    "inbound": "1"
}

for service_date in SERVICE_DATES:
    print(f"\nProcessing {service_date}...")

    # Load saved SIRI data
    siri = pd.read_csv(
        SIRIVM_ROOT / service_date / "scne_fresh_sirivm.csv",
        dtype=str
    )

    # Load saved timetable data
    trips = pd.read_csv(
        TIMETABLE_ROOT / service_date / "scne_active_trips.csv",
        dtype=str
    )

    stop_times = pd.read_csv(
        TIMETABLE_ROOT / service_date / "scne_active_stop_times.csv",
        dtype=str
    )

    stops = pd.read_csv(
        TIMETABLE_ROOT / service_date / "scne_active_stops.csv",
        dtype=str
    )

    # Load routes for route_short_name
    compact_date = service_date.replace("-", "")

    gtfs_zip = (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "timetable"
        / service_date
        / f"itm_north_east_gtfs_{compact_date}.zip"
    )

    with zipfile.ZipFile(gtfs_zip, "r") as archive:
        routes = pd.read_csv(
            archive.open("routes.txt"),
            dtype=str
        )

    # Prepare GTFS trips

    trips = trips.merge(
        routes[
            ["route_id", "route_short_name"]
        ],
        on="route_id",
        how="left"
    )

    stop_times["stop_sequence"] = pd.to_numeric(
        stop_times["stop_sequence"],
        errors="coerce"
    )

    ordered_stops = stop_times.sort_values(
        ["trip_id", "stop_sequence"]
    )

    trip_endpoints = (
        ordered_stops
        .groupby("trip_id")
        .agg(
            gtfs_origin_stop=("stop_id", "first"),
            gtfs_destination_stop=("stop_id", "last"),
            departure_time=("departure_time", "first")
        )
        .reset_index()
    )

    trips = trips.merge(
        trip_endpoints,
        on="trip_id",
        how="left"
    )

    trips["origin_time"] = trips[
        "departure_time"
    ].str[:5]

    # Prepare SIRI journeys

    siri["origin_time"] = pd.to_datetime(
        siri["origin_aimed_departure_time"],
        utc=True,
        errors="coerce"
    ).dt.strftime("%H:%M")

    siri["direction_id"] = (
        siri["direction_ref"].map(direction_map)
    )

    siri_journeys = siri[
        [
            "published_line_name",
            "direction_ref",
            "direction_id",
            "origin_time",
            "origin_ref",
            "destination_ref"
        ]
    ].drop_duplicates()

    # Match SIRI journeys to GTFS trips

    candidates = siri_journeys.merge(
        trips[
            [
                "trip_id",
                "route_id",
                "route_short_name",
                "direction_id",
                "origin_time",
                "gtfs_origin_stop",
                "gtfs_destination_stop"
            ]
        ],
        left_on=[
            "published_line_name",
            "direction_id",
            "origin_time"
        ],
        right_on=[
            "route_short_name",
            "direction_id",
            "origin_time"
        ],
        how="left"
    )

    candidates = candidates[
        (candidates["origin_ref"] == candidates["gtfs_origin_stop"])
        &
        (candidates["destination_ref"] == candidates["gtfs_destination_stop"])
    ]

    journey_map = candidates[
        [
            "published_line_name",
            "direction_ref",
            "origin_time",
            "origin_ref",
            "destination_ref",
            "trip_id",
            "route_id"
        ]
    ].drop_duplicates()

    siri = siri.merge(
        journey_map,
        on=[
            "published_line_name",
            "direction_ref",
            "origin_time",
            "origin_ref",
            "destination_ref"
        ],
        how="inner"
    )

    # Clean coordinates

    siri["latitude"] = pd.to_numeric(
        siri["latitude"],
        errors="coerce"
    )

    siri["longitude"] = pd.to_numeric(
        siri["longitude"],
        errors="coerce"
    )

    siri = siri[
        siri["latitude"].between(49, 61)
        &
        siri["longitude"].between(-8, 2)
    ].copy()

    # Prepare scheduled stops

    scheduled_stops = stop_times.merge(
        stops[
            [
                "stop_id",
                "stop_name",
                "stop_lat",
                "stop_lon"
            ]
        ],
        on="stop_id",
        how="left"
    )

    scheduled_stops["stop_lat"] = pd.to_numeric(
        scheduled_stops["stop_lat"],
        errors="coerce"
    )

    scheduled_stops["stop_lon"] = pd.to_numeric(
        scheduled_stops["stop_lon"],
        errors="coerce"
    )

    # Match GPS observations to nearest stop

    matched_parts = []

    for trip_id, siri_trip in siri.groupby("trip_id"):

        gtfs_trip = scheduled_stops[
            scheduled_stops["trip_id"] == trip_id
        ]

        if gtfs_trip.empty:
            continue

        tree = cKDTree(
            gtfs_trip[
                ["stop_lat", "stop_lon"]
            ].values
        )

        _, indexes = tree.query(
            siri_trip[
                ["latitude", "longitude"]
            ].values,
            k=1
        )

        nearest = gtfs_trip.iloc[indexes]

        siri_trip = siri_trip.copy()

        siri_trip["nearest_stop_id"] = (
            nearest["stop_id"].values
        )

        siri_trip["nearest_stop_sequence"] = (
            nearest["stop_sequence"].values
        )

        lat_diff = (
            siri_trip["latitude"].values
            - nearest["stop_lat"].values
        )

        lon_diff = (
            siri_trip["longitude"].values
            - nearest["stop_lon"].values
        )

        mean_lat = np.radians(
            (
                siri_trip["latitude"].values
                + nearest["stop_lat"].values
            ) / 2
        )

        siri_trip["stop_distance_m"] = np.sqrt(
            (lat_diff * 111320) ** 2
            +
            (
                lon_diff
                * 111320
                * np.cos(mean_lat)
            ) ** 2
        )

        matched_parts.append(siri_trip)

    matched = pd.concat(
        matched_parts,
        ignore_index=True
    )

    # Keep reliable matches
    matched = matched[
        matched["stop_distance_m"] <= 200
    ]

    # One observation per trip-stop
    matched = (
        matched
        .sort_values(
            [
                "trip_id",
                "nearest_stop_sequence",
                "stop_distance_m"
            ]
        )
        .drop_duplicates(
            subset=[
                "trip_id",
                "nearest_stop_sequence"
            ]
        )
        .copy()
    )

    # Add scheduled arrival time

    scheduled_lookup = scheduled_stops[
        [
            "trip_id",
            "stop_sequence",
            "stop_id",
            "stop_name",
            "arrival_time"
        ]
    ]

    matched = matched.merge(
        scheduled_lookup,
        left_on=[
            "trip_id",
            "nearest_stop_sequence"
        ],
        right_on=[
            "trip_id",
            "stop_sequence"
        ],
        how="left"
    )

    matched["actual_observed_time"] = pd.to_datetime(
        matched["recorded_at_time"],
        utc=True,
        errors="coerce"
    )

    def scheduled_datetime(gtfs_time):
        if pd.isna(gtfs_time):
            return pd.NaT

        h, m, s = map(
            int,
            gtfs_time.split(":")
        )

        return (
            pd.Timestamp(
                service_date,
                tz="UTC"
            )
            + timedelta(
                hours=h,
                minutes=m,
                seconds=s
            )
        )

    matched["scheduled_arrival_time"] = (
        matched["arrival_time"]
        .apply(scheduled_datetime)
    )

    # Create regression target

    matched["delay_seconds"] = (
        matched["actual_observed_time"]
        - matched["scheduled_arrival_time"]
    ).dt.total_seconds()

    matched = matched[
        matched["delay_seconds"].between(
            -900,
            1800
        )
    ].copy()

    # Save final date dataset

    output_file = (
        CLEAN_OUTPUT_ROOT
        / f"scne_delay_{service_date}.csv"
    )

    matched.to_csv(
        output_file,
        index=False
    )

    print("Final rows:", len(matched))
    print("Unique trips:", matched["trip_id"].nunique())
    print("Saved:", output_file)


Processing 2025-12-26...
Final rows: 31891
Unique trips: 887
Saved: D:\Big Data Programming Project\Final Assignment\data\processed\delay_data\scne_delay_2025-12-26.csv

Processing 2025-12-27...
Final rows: 170149
Unique trips: 4690
Saved: D:\Big Data Programming Project\Final Assignment\data\processed\delay_data\scne_delay_2025-12-27.csv

Processing 2025-12-28...
Final rows: 106845
Unique trips: 2949
Saved: D:\Big Data Programming Project\Final Assignment\data\processed\delay_data\scne_delay_2025-12-28.csv


In [27]:
# Final validation for Notebook 03

total_rows = 0

for service_date in SERVICE_DATES:
    file_path = (
        CLEAN_OUTPUT_ROOT
        / f"scne_delay_{service_date}.csv"
    )

    df = pd.read_csv(file_path)

    total_rows += len(df)

    print(f"\nDate: {service_date}")
    print("Rows:", len(df))
    print("Unique trips:", df["trip_id"].nunique())
    print("Missing delay:", df["delay_seconds"].isna().sum())
    print(
        "Duplicate trip-stop rows:",
        df.duplicated(
            subset=["trip_id", "stop_sequence"]
        ).sum()
    )
    print(
        "Delay range:",
        df["delay_seconds"].min(),
        "to",
        df["delay_seconds"].max()
    )

print("\nTotal cleaned delay rows:", total_rows)


Date: 2025-12-26
Rows: 31891
Unique trips: 887
Missing delay: 0
Duplicate trip-stop rows: 0
Delay range: -899.0 to 1800.0

Date: 2025-12-27
Rows: 170149
Unique trips: 4690
Missing delay: 0
Duplicate trip-stop rows: 0
Delay range: -899.0 to 1800.0

Date: 2025-12-28
Rows: 106845
Unique trips: 2949
Missing delay: 0
Duplicate trip-stop rows: 0
Delay range: -900.0 to 1800.0

Total cleaned delay rows: 308885
